In [ ]:
import numpy as np
import scipy.io
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
import os

# ✅ 1. MAT 파일 로드 (기존 데이터 학습)
nominal_data = scipy.io.loadmat("ACF_Nominal.mat")["ACF_Nominal"]
tma_data = scipy.io.loadmat("ACF_TMA.mat")["ACF_TMA"]
tmb_data = scipy.io.loadmat("ACF_TMB.mat")["ACF_TMB"]
tmc_data = scipy.io.loadmat("ACF_TMC.mat")["ACF_TMC"]

# ✅ 2. 데이터 평탄화 (Flatten)
nominal_flat = nominal_data.reshape(nominal_data.shape[0], -1)
tma_flat = tma_data.reshape(-1, nominal_flat.shape[1])
tmb_flat = tmb_data.reshape(-1, nominal_flat.shape[1])
tmc_flat = tmc_data.reshape(-1, nominal_flat.shape[1])

# ✅ 3. 데이터 병합 (정상: 0, 비정상: 1)
X = np.vstack((nominal_flat, tma_flat, tmb_flat, tmc_flat))
y = np.concatenate((
    np.zeros(nominal_flat.shape[0]),  # 정상 데이터 (0)
    np.ones(tma_flat.shape[0]),  # 비정상 데이터 (1)
    np.ones(tmb_flat.shape[0]),
    np.ones(tmc_flat.shape[0])
))

# ✅ 4. 로그 변환 및 정규화
X = np.log1p(X)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# ✅ 5. 정상 데이터 Oversampling (SMOTE 적용)
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# ✅ 6. 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# ✅ 7. 랜덤 포레스트 모델 학습
rf_clf_optimized = RandomForestClassifier(
    n_estimators=80,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42
)
rf_clf_optimized.fit(X_train, y_train)

# ✅ 8. 새로운 데이터 파일 로드 함수
def load_new_data(file_path):
    """
    새로운 데이터 파일(.mat)을 로드하여 변환
    """
    mat_data = scipy.io.loadmat(file_path)
    key = list(mat_data.keys())[-1]  # 마지막 키를 데이터로 사용 (유동적인 파일 지원)
    new_data = mat_data[key]
    new_data_flat = new_data.reshape(new_data.shape[0], -1)
    return new_data_flat

# ✅ 9. 새로운 데이터 예측 함수
def predict_new_file(file_path, ranging_error):
    """
    새로운 데이터 파일을 불러와 예측 수행
    """
    new_data = load_new_data(file_path)
    new_data = np.log1p(new_data)  # 로그 변환
    new_data = scaler.transform(new_data)  # 정규화
    
    # 예측 수행
    probs = rf_clf_optimized.predict_proba(new_data)[:, 1]
    predictions = (probs >= 0.75).astype(int)  # 최적 Threshold 적용
    
    # 결과 출력
    for i, (prob, pred) in enumerate(zip(probs, predictions)):
        print(f"샘플 {i+1}: 예측 결과: {'비정상(1)' if pred == 1 else '정상(0)'} (확률: {prob:.2f}) | Ranging Error: {ranging_error:.2f} [m]")

# ✅ 10. 새로운 데이터 예측 실행 예제
new_file_path = "new_data.mat"  # 새로운 데이터 파일 경로
if os.path.exists(new_file_path):
    predict_new_file(new_file_path, ranging_error=2.5)  # 예제 Ranging Error 값 적용
